# Experiment 1: LIME Hyperparameter Tuning Analysis

**User Story:** EXP1-31  
**Goal:** Analyze the results of the LIME parameter sweep to find a configuration that maximizes Fidelity ($R^2 > 0.5$) and Stability.

## 1. Setup

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

: 

## 2. Load Data
Loading results from `outputs/lime_tuning_results.csv`.

In [ ]:
results_path = Path("../outputs/lime_tuning_results.csv")

if not results_path.exists():
    print(f"Warning: {results_path} not found. Please run scripts/run_batch_experiments.py first.")
    # Creating mock data for visualization structure demonstration
    print("Generating MOCK data for demonstration...")
    data = []
    for kw in [0.25, 0.5, 0.75, 1.0, 2.0, "auto"]:
        for ns in [1000, 5000]:
            # Mocking correlations: Higher NS -> Better Stability, Local Fidelity varies
            kw_val = 0.75 if kw == "auto" else kw
            
            fidelity = 0.4 + (0.1 * np.random.randn()) # Bad fidelity generally
            if kw == 0.75 or kw == "auto":
                fidelity += 0.3 # Sweet spot
            
            stability = 0.5 + (0.2 if ns == 5000 else 0.0)
            
            data.append({
                "experiment_name": f"exp1_lime_k{kw}_n{ns}",
                "kernel_width": str(kw),
                "num_samples": ns,
                "fidelity_mean": fidelity,
                "stability_mean": stability,
                "cost_mean": 100 if ns == 1000 else 500
            })
    df = pd.DataFrame(data)
else:
    df = pd.read_csv(results_path)
    
    # Parse params from experiment name if not in columns, though our flat csv should have them if we added them.
    # But the batch runner currently only flattens metrics and top-level metadata.
    # We need to extract K and N from experiment_name string: "exp1_lime_k{kw}_n{ns}_fs{fs}"
    def parse_name(name):
        try:
            parts = name.split('_')
            # parts: exp1, lime, k..., n..., fs...
            k_part = [p for p in parts if p.startswith('k')][0]
            n_part = [p for p in parts if p.startswith('n')][0]
            
            k_val = k_part[1:]
            n_val = int(n_part[1:])
            return k_val, n_val
        except:
            return None, None

    df[['kernel_width', 'num_samples']] = df['experiment_name'].apply(
        lambda x: pd.Series(parse_name(x))
    )

df.head()

## 3. Fidelity Analysis
We calculate the mean R2 score (Faithfulness) for each Kernel Width.

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='kernel_width', y='fidelity_mean', hue='num_samples')
plt.title("LIME Fidelity (R2) vs Kernel Width")
plt.ylabel("Mean R2 Score")
plt.xlabel("Kernel Width")
plt.axhline(0.5, color='r', linestyle='--', label='Min Threshold (0.5)')
plt.legend()
plt.show()

## 4. Stability Analysis
Checking consistency of explanations.

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x='kernel_width', y='stability_mean', hue='num_samples', marker='o')
plt.title("LIME Stability vs Kernel Width")
plt.ylabel("Cosine Similarity")
plt.show()

## 5. Conclusion
Identifying the best configuration.

In [ ]:
# Filter for Fidelity > 0.5
viable = df[df['fidelity_mean'] > 0.5].copy()

if not viable.empty:
    # Sort by Stability descending
    best = viable.sort_values('stability_mean', ascending=False).iloc[0]
    print("Recommended Configuration:")
    print(f"Kernel Width: {best['kernel_width']}")
    print(f"Num Samples: {best['num_samples']}")
    print(f"Fidelity: {best['fidelity_mean']:.3f}")
    print(f"Stability: {best['stability_mean']:.3f}")
else:
    print("No configuration met the R2 > 0.5 threshold.")
    print("Top configuration by Fidelity:")
    best_fidelity = df.sort_values('fidelity_mean', ascending=False).iloc[0]
    print(f"Kernel Width: {best_fidelity['kernel_width']}")
    print(f"Fidelity: {best_fidelity['fidelity_mean']:.3f}")